### Validate different types of errors using MAP as an example

#### Invalid values or out of range

In the following example, several fields are mutated to have invalid values or with values out of range. The validator collects all errors and pinpoint the exact field that is invalid, with nested key for dictionary and indexing for list type.

In [5]:
import copy
from pprint import pprint
import yaml
from v2x_msg_validator.validator import V2XMessageValidator


def read_sample_yaml(filepath):
    with open(filepath, "r") as file:
        data = yaml.safe_load(file)
    return data

data = read_sample_yaml("../test/j2735_202409/fixtures/MAP.yaml")

# mutate data
_data = copy.deepcopy(data)
_data["value"]["MapData"]["intersections"][0]["refPoint"]["lat"] = "not_a_number"
_data["value"]["MapData"]["intersections"][0]["laneSet"][0]["laneAttributes"]["directionalUse"] = "08"
_data["value"]["MapData"]["intersections"][0]["laneSet"][0]["nodeList"]["nodes"][0]["delta"]["node-XY6"]["y"] = "bad"

validator = V2XMessageValidator(revision="j2735_202409")
errors = validator.validate(_data)

pprint(errors)



[ASN1ObjValErr[key='MapData.intersections[0].refPoint.lat', code=0, msg='invalid named value', val='not_a_number'],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].nodeList.nodes[0].delta.node-XY6.y', code=0, msg='invalid named value', val='bad'],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].laneAttributes.directionalUse', code=3, msg='value out of size constraint', val=(8, 8)]]


#### Missing keys

A required key is deleted: `msgIssueRevision`.

In [6]:
_data = copy.deepcopy(data)
del _data["value"]["MapData"]["msgIssueRevision"]

errors = validator.validate(_data)
pprint(errors)

[ASN1ObjValErr[key='MapData', code=1, msg="missing mandatory value(s): {'msgIssueRevision'}", val={'timeStamp': 400000, 'layerType': 'intersectionData', 'layerID': 1, 'intersections': [{'name': 'TestIntersection', 'id': {'region': 100, 'id': 1001}, 'revision': 3, 'refPoint': {'lat': 389284111, 'long': -772410713, 'elevation': 1000}, 'laneWidth': 366, 'speedLimits': [{'type': 'maxSpeedInSchoolZone', 'speed': 500}], 'laneSet': [{'laneID': 1, 'name': 'Lane1', 'ingressApproach': 1, 'egressApproach': 2, 'laneAttributes': {'directionalUse': '10', 'sharedWith': '0000000000', 'laneType': {'vehicle': '00000000'}}, 'maneuvers': '800', 'nodeList': {'nodes': [{'delta': {'node-XY6': {'x': -4012, 'y': 365}}, 'attributes': {'localNode': ['stopLine'], 'disabled': ['doNotBlock'], 'enabled': ['adaptiveTimingPresent'], 'data': [{'pathEndPointAngle': 10}], 'dWidth': 50, 'dElevation': 10}}, {'delta': {'node-XY6': {'x': -5541, 'y': 7249}}}]}, 'connectsTo': [{'connectingLane': {'lane': 25, 'maneuver': '800'}

#### Additional key

An `extra_key` is added which does not belong in the schema.

In [7]:
_data = copy.deepcopy(data)
_data["value"]["MapData"]["extra_key"] = "some_value"
errors = validator.validate(_data)
pprint(errors)

[ASN1ObjValErr[key='MapData', code=2, msg='invalid key: extra_key', val={'timeStamp': 400000, 'msgIssueRevision': 5, 'layerType': 'intersectionData', 'layerID': 1, 'intersections': [{'name': 'TestIntersection', 'id': {'region': 100, 'id': 1001}, 'revision': 3, 'refPoint': {'lat': 389284111, 'long': -772410713, 'elevation': 1000}, 'laneWidth': 366, 'speedLimits': [{'type': 'maxSpeedInSchoolZone', 'speed': 500}], 'laneSet': [{'laneID': 1, 'name': 'Lane1', 'ingressApproach': 1, 'egressApproach': 2, 'laneAttributes': {'directionalUse': '10', 'sharedWith': '0000000000', 'laneType': {'vehicle': '00000000'}}, 'maneuvers': '800', 'nodeList': {'nodes': [{'delta': {'node-XY6': {'x': -4012, 'y': 365}}, 'attributes': {'localNode': ['stopLine'], 'disabled': ['doNotBlock'], 'enabled': ['adaptiveTimingPresent'], 'data': [{'pathEndPointAngle': 10}], 'dWidth': 50, 'dElevation': 10}}, {'delta': {'node-XY6': {'x': -5541, 'y': 7249}}}]}, 'connectsTo': [{'connectingLane': {'lane': 25, 'maneuver': '800'}, '

### Validate SPaT message

Validating different message types share the same API.

In [10]:
data = read_sample_yaml("../samples/sampleSPAT.yaml")
pprint(data)

{'messageId': 19,
 'value': {'SPAT': {'intersections': [{'id': {'id': 9709},
                                       'moy': 428788,
                                       'name': 'West Intersection',
                                       'revision': 1,
                                       'states': [{'signalGroup': 1,
                                                   'state-time-speed': [{'eventState': 'stop-And-Remain',
                                                                         'timing': {'minEndTime': 17261}}]},
                                                  {'signalGroup': 2,
                                                   'state-time-speed': [{'eventState': 'protected-Movement-Allowed',
                                                                         'timing': {'maxEndTime': 17399,
                                                                                    'minEndTime': 17399}}]},
                                                  {'signalGroup

In [11]:
errors = validator.validate(data)
pprint(errors)

[]


### Validate TravelerInformation message

In [12]:
data = read_sample_yaml("../samples/sampleTIM.yaml")
pprint(data)

{'messageId': 31,
 'value': {'TravelerInformation': {'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                                                              {'item': {'text': 'Curve '
                                                                                                'Ahead'}},
                                                                              {'item': {'itis': 2564}},
                                                                              {'item': {'text': '25'}},
                                                                              {'item': {'itis': 8720}}]},
                                                   'contentNew': {'frictionInfo': {'roadSurfaceDescription': {'portlandCement': {'type': 'newSharp'}}}},
                                                   'doNotUse1': 0,
                                                   'doNotUse2': 0,
                                                   'doNotUse3': 0,


In [13]:
errors = validator.validate(data)
pprint(errors)

[]


The validated message is encoded with UPER and subsequently decoded, which match the initial data.

In [14]:
import json

from v2x_codecs import j2735_202409

TIM = j2735_202409.TravelerInformation.TravelerInformation
# convert to uper encoding, decode, and compare against original data
buf = TIM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
TIM.from_uper(buf)

# convert to json
data_decoded = json.loads(TIM.to_json())

print("==== decoded json ===")
pprint(data_decoded)


assert data["value"]["TravelerInformation"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'\x00\x11\x00b\x99\xb9\xea\xc2z\x9b\xaat#!X \xc0\xa9\xdf@(\x00~S7=XOSuN\x84'
 b'd\x00\xb7!\x00\x00\x00\xb8\xf57N6f\xe7\xacP\x13\xec\xe3\xd4\xdd\xc1\t\x9b'
 b'\x9e\x98\x80PS\x8fSx\xf9fnzZ\x81A@\x03@\x00\xde\xa1\xf5\xe5\xdb*\x08:2'
 b'\xe1\xc8\n\x04\x8b&\xa2!\x00\x10 \x00\x00')
==== decoded json ===
{'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                            {'item': {'text': 'Curve Ahead'}},
                                            {'item': {'itis': 2564}},
                                            {'item': {'text': '25'}},
                                            {'item': {'itis': 8720}}]},
                 'contentNew': {'frictionInfo': {'roadSurfaceDescription': {'portlandCement': {'type': 'newSharp'}}}},
                 'doNotUse1': 0,
                 'doNotUse2': 0,
                 'doNotUse3': 0,
                 'doNotUse4': 0,
                 'durationTime': 32000,
                 'frameTyp

### Validate PersonalSafetyMessage

In [15]:
data = read_sample_yaml("../samples/samplePSM.yaml")
pprint(data)

{'messageId': 32,
 'value': {'PersonalSafetyMessage': {'accelSet': {'lat': 0,
                                                  'long': 0,
                                                  'vert': 0,
                                                  'yaw': 0},
                                     'accuracy': {'orientation': 0,
                                                  'semiMajor': 50,
                                                  'semiMinor': 50},
                                     'basicType': 'aPEDESTRIAN',
                                     'heading': 8000,
                                     'id': '00000001',
                                     'msgCnt': 1,
                                     'pathPrediction': {'confidence': 200,
                                                        'radiusOfCurve': 32767},
                                     'position': {'elevation': 400,
                                                  'lat': 389549153,
                    

In [16]:
errors = validator.validate(data)
pprint(errors)

[]


In [17]:
from v2x_codecs import j2735_202409

PSM = j2735_202409.PersonalSafetyMessage.PersonalSafetyMessage
# convert to uper encoding, decode, and compare against original data
buf = PSM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
PSM.from_uper(buf)

# convert to json
data_decoded = json.loads(PSM.to_json())

print("==== decoded json ===")
pprint(data_decoded)

assert data["value"]["PersonalSafetyMessage"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'X\x00\x03_\x90\x04\x00\x00\x00\x05L\xdc\xf5a=M\xd5:\x11\x9022\x00\x00'
 b'\x03\xe9\xf4\x07\xd0}\x07\xf7\xff\xf7\xff\xf6@ ')
==== decoded json ===
{'accelSet': {'lat': 0, 'long': 0, 'vert': 0, 'yaw': 0},
 'accuracy': {'orientation': 0, 'semiMajor': 50, 'semiMinor': 50},
 'basicType': 'aPEDESTRIAN',
 'heading': 8000,
 'id': '00000001',
 'msgCnt': 1,
 'pathPrediction': {'confidence': 200, 'radiusOfCurve': 32767},
 'position': {'elevation': 400, 'lat': 389549153, 'long': -771488965},
 'propulsion': {'human': 'onFoot'},
 'secMark': 45000,
 'speed': 125}
Decoded data match initial data
